# 04 — Dual-Attention BiGRU on UCI-HAR

Week 7 milestone: the proposed model is complete, and **all 4 models now run
on UCI-HAR under both evaluation protocols** (official split + subject-
independent LOSO), 3 seeds each. This notebook is the first place the full
comparison exists end to end, plus a **first significance test** (Wilcoxon
signed-rank, per-fold, Dual-Attention BiGRU vs. each baseline).

This is a rehearsal for the master comparison across all 3 datasets
(Weeks 8-9) — the code here is dataset-agnostic (just swap `DATASET`) and
should need no changes to run on WISDM or PAMAP2 later.

Prerequisite: `python -m src.preprocessing` has populated
`data/processed/ucihar/`.


## Setup

In [ ]:
import sys, os
sys.path.append("..")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # so relative paths in src/*.py (data/, checkpoints/, results/) resolve correctly
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import wilcoxon

from src.data_loader import get_official_split, get_loso_folds, build_dataloaders
from src.train import MODEL_REGISTRY, build_model, train_one_run
from src.utils import set_seed, get_device, encode_labels, profile_model, append_result_row

DATASET = "ucihar"
SEEDS = [0, 1, 2]           # minimum per plan's evaluation protocol
EPOCHS = 50
PATIENCE = 8
BATCH_SIZE = 64
RESULTS_CSV = Path("../results/tables/raw_results.csv")

device = get_device()
print("device:", device)
print("registered models:", list(MODEL_REGISTRY.keys()))
assert len(MODEL_REGISTRY) == 4, "expected all 4 models to be registered by Week 7"


## 1. Official split — all 4 models, 3 seeds each

Efficiency (params/FLOPs/latency) is computed once per architecture and
attached to every row for that model — it doesn't depend on trained weights.


In [ ]:
for model_name in MODEL_REGISTRY:
    X_train, y_train, X_test, y_test = get_official_split(DATASET)
    y_train_enc, y_test_enc, mapping = encode_labels(y_train, y_test)
    input_shape = X_train.shape[1:]

    probe = build_model(model_name, input_size=input_shape[-1], num_classes=len(mapping))
    efficiency = profile_model(probe, input_shape, device)
    print(f"{model_name:28s} params={efficiency['params']:>8} flops={efficiency['flops']:.2e}")

    for seed in SEEDS:
        set_seed(seed)
        model = build_model(model_name, input_size=input_shape[-1], num_classes=len(mapping))
        train_loader, test_loader = build_dataloaders(
            X_train, y_train_enc, X_test, y_test_enc, batch_size=BATCH_SIZE,
        )
        metrics = train_one_run(model, train_loader, test_loader, device,
                                 epochs=EPOCHS, lr=1e-3, patience=PATIENCE)
        append_result_row({
            "model": model_name, "dataset": DATASET, "split_mode": "official",
            "fold_id": None, "seed": seed, **metrics, **efficiency,
        }, csv_path=RESULTS_CSV)
    print(f"  {model_name} official split done ({len(SEEDS)} seeds)\n")


## 2. Subject-independent LOSO — all 4 models, all folds, 3 seeds each

This is the headline evaluation protocol. Each fold holds out a distinct
group of subjects entirely; normalization is fit on that fold's train
subjects only (see `data_loader.py::get_loso_folds`).

This is the slow cell — 4 models × `LOSO_N_FOLDS["ucihar"]` folds × 3 seeds.


In [ ]:
for model_name in MODEL_REGISTRY:
    print(f"=== {model_name} ===")
    for fold in get_loso_folds(DATASET):
        fold_id = fold["fold_id"]
        y_train_enc, y_test_enc, mapping = encode_labels(fold["y_train"], fold["y_test"])
        input_shape = fold["X_train"].shape[1:]

        probe = build_model(model_name, input_size=input_shape[-1], num_classes=len(mapping))
        efficiency = profile_model(probe, input_shape, device)

        for seed in SEEDS:
            set_seed(seed)
            model = build_model(model_name, input_size=input_shape[-1], num_classes=len(mapping))
            train_loader, test_loader = build_dataloaders(
                fold["X_train"], y_train_enc, fold["X_test"], y_test_enc, batch_size=BATCH_SIZE,
            )
            metrics = train_one_run(model, train_loader, test_loader, device,
                                     epochs=EPOCHS, lr=1e-3, patience=PATIENCE)
            append_result_row({
                "model": model_name, "dataset": DATASET, "split_mode": "loso",
                "fold_id": fold_id, "seed": seed, **metrics, **efficiency,
            }, csv_path=RESULTS_CSV)
        print(f"  fold {fold_id} (held out {fold['held_out_subjects']}) done")


## 3. Aggregate results

Mean ± std accuracy and macro-F1 per model, per split mode, across seeds
(official) or across seeds x folds (LOSO).

In [ ]:
df = pd.read_csv(RESULTS_CSV)
df = df[df["dataset"] == DATASET]

summary = df.groupby(["model", "split_mode"])[["accuracy", "macro_f1"]].agg(["mean", "std"])
summary


## 4. Efficiency comparison

Params / FLOPs / latency per model (identical across seeds/folds for a
given architecture, so one row per model is enough here).

In [ ]:
efficiency_cols = ["model", "params", "flops", "latency_ms_bs1", "latency_ms_bs32"]
eff_summary = df.drop_duplicates(subset="model")[efficiency_cols].set_index("model")
eff_summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
acc_mean = df[df.split_mode == "loso"].groupby("model")["accuracy"].mean()
for model_name in acc_mean.index:
    ax.scatter(eff_summary.loc[model_name, "params"], acc_mean[model_name], s=80, label=model_name)
ax.set_xlabel("parameters")
ax.set_ylabel("LOSO mean accuracy")
ax.set_title(f"{DATASET} — accuracy vs. efficiency (LOSO)")
ax.legend()
plt.tight_layout()
plt.show()


## 5. First significance test (Wilcoxon signed-rank)

Per-fold accuracy (averaged across the 3 seeds within each fold) gives one
paired sample per fold. Wilcoxon signed-rank compares Dual-Attention BiGRU's
per-fold scores against each baseline's per-fold scores on the *same* folds
(same held-out subjects), which is the correct paired test here — an
unpaired test would ignore that folds share held-out subjects across models.

Caveat: with only `LOSO_N_FOLDS["ucihar"]` folds, statistical power is
limited — treat p-values here as a first look, not a final claim. The full
version of this (with effect sizes, multiple-comparison handling if needed,
and results across all 3 datasets) belongs in `src/stats_test.py` (Week 9).


In [ ]:
loso = df[df.split_mode == "loso"]
per_fold = loso.groupby(["model", "fold_id"])["accuracy"].mean().reset_index()
pivot = per_fold.pivot(index="fold_id", columns="model", values="accuracy")
display(pivot)

target = "dual_attention_bigru"
for baseline in [m for m in MODEL_REGISTRY if m != target]:
    stat, p = wilcoxon(pivot[target], pivot[baseline])
    direction = "higher" if pivot[target].mean() > pivot[baseline].mean() else "lower"
    print(f"{target} vs {baseline:28s} mean diff={pivot[target].mean()-pivot[baseline].mean():+.4f} "
          f"({direction})  Wilcoxon p={p:.4f}")


## 6. Attention weight inspection

Qualitative look at what the trained Dual-Attention BiGRU actually attends
to — channel attention (which of the 6 sensor-derived feature channels get
up-weighted) and temporal attention (which timesteps within a 2.56s window
matter most). This previews the ablation study (channel-only / temporal-only
/ neither variants) that isolates each attention type's contribution.


In [ ]:
X_train, y_train, X_test, y_test = get_official_split(DATASET)
y_train_enc, y_test_enc, mapping = encode_labels(y_train, y_test)

set_seed(0)
model = build_model("dual_attention_bigru", input_size=X_train.shape[-1], num_classes=len(mapping))
train_loader, test_loader = build_dataloaders(X_train, y_train_enc, X_test, y_test_enc, batch_size=BATCH_SIZE)
train_one_run(model, train_loader, test_loader, device, epochs=EPOCHS, lr=1e-3, patience=PATIENCE)

import torch
model.eval()
sample_x, sample_y = next(iter(test_loader))
with torch.no_grad():
    logits, attn = model(sample_x.to(device), return_attention=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(attn["channel_weights"].shape[1]), attn["channel_weights"][0].cpu().numpy())
axes[0].set_title("Channel attention weights (sample 0)")
axes[0].set_xlabel("hidden feature channel")

axes[1].plot(attn["temporal_weights"][0].cpu().numpy())
axes[1].set_title("Temporal attention weights (sample 0)")
axes[1].set_xlabel("timestep")
plt.tight_layout()
plt.show()


## Notes / next steps

- [ ] If Dual-Attention BiGRU's LOSO accuracy is within noise of the CNN
      reference (§5 p-values not significant, §4 scatter close together),
      that's still a usable headline result per the plan (efficiency win at
      comparable accuracy) — don't need it to be strictly *better*.
- [ ] Ablation runs (channel-only / temporal-only / neither) not yet done
      here — `DualAttentionBiGRU` supports `use_channel_attention` /
      `use_temporal_attention` flags directly; add a small loop over these
      four configs before Week 9's master table if ablation results are
      wanted in the paper.
- [ ] Repeat §1–§5 verbatim on WISDM and PAMAP2 (Weeks 8–9) — only
      `DATASET` and `LOSO_N_FOLDS` differ, no code changes expected.
- [ ] Move the Wilcoxon logic in §5 into `src/stats_test.py` once all 3
      datasets are in, so the final `master_comparison.csv` step can call it
      directly rather than duplicating it per-notebook.
